# KAN-89 (E12) Gradient boosting: pooled LightGBM on fs_v2_core / fs_v2_full

Eight sections of protocol section 13. Every number below is printed by code from the MLflow store (dev split only; the holdout was never touched).

In [ ]:
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

from pm25_forecast.experiments import gbm_summary as gs
from pm25_forecast.experiments import tracking
from pm25_forecast.experiments.prereg import load_prereg

# Parameters: edit only this cell.
TICKET = "KAN-89"
BEST_PREREG = "e12-full-l2-delta"
SNAPSHOT_DIR = "~/pm25-experiments/snapshots/e02-v1"
LOG_DIR = "out/kan89"
rows = gs._runs()
prim = [r for r in rows if r["id"].startswith(("e12-core", "e12-full"))]
abl = [r for r in rows if r["id"].startswith("e12-abl")]
best = min(prim, key=lambda r: r["mae"])
assert best["id"] == BEST_PREREG, best["id"]

## 1. Summary

Question: does a pooled, tuned LightGBM on the v2 feature sets beat persistence and the smoother at +24 h on the dev folds? The cell below prints the answer; the decision follows the preregistered rule.

In [ ]:
smoother, pers = rows[0]["smoother_mae"], rows[0]["persistence_mae"]
lo, hi = best["d_s_ci"]
print(
    f"best of {len(prim)} primary variants: {best['id']}  mae_24h={best['mae']:.3f}  "
    f"smoother={smoother:.3f}  persistence={pers:.3f}"
)
print(
    f"diff vs smoother {best['d_s']:+.3f} [{lo:+.3f}, {hi:+.3f}] (95%, unadjusted)  "
    f"beats smoother (CI high < 0): {hi < 0}"
)
lo, hi = best["d_p_ci"]
print(f"diff vs persistence {best['d_p']:+.3f} [{lo:+.3f}, {hi:+.3f}]  beats persistence: {hi < 0}")
print(
    "variants whose CI vs smoother is entirely below zero:",
    [r["id"] for r in rows if r["d_s_ci"][1] < 0],
)

## 2. Hypothesis and decision rule

From the committed preregistration files (printed, not edited).

In [ ]:
p = load_prereg(f"docs/experiments/prereg/{BEST_PREREG}.yaml")
print(p.hypothesis)
print(p.decision_rule)

## 3. Setup

Frozen snapshot e02-v1, air4thai era, 5 weekly dev folds, weekly refit at fold start minus 24 h, expanding window. Features from `library_v2` (`prepare_series` once per fold). One pooled LightGBM per horizon, station as a categorical feature. Tuning: 8 configurations (default plus 7 seeded random draws), 3 inner expanding-window folds on the training part only, purged by the horizon.

In [ ]:
print(tracking.tracking_uri(), tracking.snapshot_hash(Path(SNAPSHOT_DIR).expanduser()))
print(
    "dirty runs:", [r["id"] for r in rows if r["dirty"] != "false"], " CI level:", best["ci_level"]
)

## 4. EDA and data checks

Coverage and exclusions are enforced by the runner (`evaluation.json`: coverage, eligible pairs, serving lag). Cross-station features are evaluated at origin minus 1 h because the predictor sees one station per call (see the module docstring of `gbm.py`).

In [ ]:
import json

cl = tracking._client()
ev = json.loads(Path(cl.download_artifacts(best["run_id"], "evaluation.json")).read_text())
s = ev["scores"]["24"]
print(
    {k: s[k] for k in ("coverage", "eligible_pairs", "n_pairs", "n_origins", "grid_hours")},
    ev["invalid_reasons"],
    ev["counters"],
)

## 5. Results table

Primary variants (12) and the ablation on the best (4). 16 dev runs in total; CIs are unadjusted.

In [ ]:
print(gs.table(prim))
print(gs.table([best, *abl]))

## 6. Statistical tests and diagnostics

Moving-block bootstrap CI (72 h blocks), sensitivities, Diebold-Mariano against the smoother, per-fold diffs for the best run.

In [ ]:
print(
    "CI sensitivity vs smoother:",
    s["vs_smoother"]["ci_sensitivity"],
    "fragile:",
    s["vs_smoother"]["fragile"],
)
print("DM vs smoother:", s["vs_smoother"]["dm"])
for k, f in s["by_fold"].items():
    print(
        "fold",
        k,
        f"vs pers {f['diff_vs_persistence']:+.3f} vs smoother {f['diff_vs_smoother']:+.3f}",
    )
for st, v in s["mae_by_station"].items():
    print(st, v)

## 7. Figures

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
allr = sorted(prim + abl, key=lambda r: r["mae"])
ax.barh([r["id"] for r in allr], [r["mae"] for r in allr], color="#4c78a8")
ax.axvline(smoother, color="k", ls="--", label="smoother")
ax.axvline(pers, color="r", ls=":", label="persistence")
ax.set_xlim(3.7, pers + 0.1)
ax.set_xlabel("mae_24h (ug/m3), dev")
ax.legend()
plt.tight_layout()
plt.show()
print(gs.tuning_tables(Path(LOG_DIR), BEST_PREREG))

## 8. Conclusion, limitations, next

Compare the printed CI with the decision rule: a variant beats a baseline only if its CI high is below zero. Limits: dev only (5 weekly folds, 5 stations, one season), 16 unadjusted dev runs with a best-of-12 selection, no holdout. The runner's one-station predictor forced the cross-station group to use a 1 h older peer reading and no nearest-station feature.